# Can we link client and service traffic from the two Guards? (minimal analysis)

**Question.** An observer sees only the Guard of the client and the Guard of the onion service. Can it tell, by timing alone, which traffic at one Guard belongs to which traffic at the other?

**Method.**
1. Group the packets of each Guard into *bursts* (same direction, gap <= 0.1 s) and keep bursts >= 1000 bytes. Only their start time is used.
2. Pair each client burst with the closest unused service burst starting within +/-0.05 s (one-to-one).
3. **Control:** do the same with the service trace of a *different* run (aligned to the start of the run). No real match is possible there, so it shows how many pairs appear by chance.
4. Count how many of the 100 requests have at least one pair.

If the real run pairs far more than the control, the two Guards can be linked by timing.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Configuration (change when the network is recreated) ----------------
CLIENT_GUARD_IP  = "10.5.184.229"
SERVICE_GUARD_IP = "10.5.237.145"
RUNS = ["run_01", "run_02", "run_03", "run_04", "run_05"]
BURST_GAP, MIN_BYTES, LAG = 0.1, 1000, 0.05      # seconds, bytes, seconds

RESULTS = next(p for p in [Path("results"), Path("experiments/onion-correlation/results")] if p.is_dir())

def burst_starts(csv_path, guard_ip, other_guard_ip):
    """Start times of the bursts of one Guard."""
    df = pd.read_csv(csv_path)
    df.columns = ["time", "size", "src", "dst"]
    df = df.dropna(subset=["src", "dst"]).sort_values("time")
    peer = np.where(df["src"] == guard_ip, df["dst"], df["src"])
    df = df[peer != other_guard_ip].copy()        # the two Guards also talk to each other: those packets are in both captures, remove them
    df["dir"] = np.where(df["src"] == guard_ip, 1, -1)
    df["burst"] = ((df["time"].diff() > BURST_GAP) | (df["dir"] != df["dir"].shift())).cumsum()
    b = df.groupby("burst").agg(start=("time", "min"), bytes=("size", "sum"))
    return np.sort(b.loc[b["bytes"] >= MIN_BYTES, "start"].to_numpy())

client, service, requests = {}, {}, {}
for run in RUNS:
    folder = RESULTS / run / "analysis" / "window_5s"
    client[run]  = burst_starts(folder / "client_metadata.csv",  CLIENT_GUARD_IP,  SERVICE_GUARD_IP)
    service[run] = burst_starts(folder / "service_metadata.csv", SERVICE_GUARD_IP, CLIENT_GUARD_IP)
    req = pd.read_csv(RESULTS / run / "client_requests_log.csv")
    requests[run] = req[req["success"]]
    print(run, "client bursts:", len(client[run]), "| service bursts:", len(service[run]))

## Pairing and request coverage

In [ ]:
def pair(client_starts, service_starts, shift=0.0):
    """One-to-one: each client burst takes the closest unused service burst within +/-LAG."""
    svc = np.sort(service_starts) + shift
    used = np.zeros(len(svc), dtype=bool)
    paired = []
    for t in client_starts:
        lo = np.searchsorted(svc, t - LAG, side="left")
        hi = np.searchsorted(svc, t + LAG, side="right")
        free = np.arange(lo, hi)[~used[lo:hi]]
        if free.size:
            j = free[np.argmin(np.abs(svc[free] - t))]
            used[j] = True
            paired.append(t)
    return np.array(paired)

def covered(paired, req):
    """Number of requests with at least one pair inside [start - LAG, end + LAG]."""
    lo = np.searchsorted(paired, req["timestamp_start"].to_numpy() - LAG, side="left")
    hi = np.searchsorted(paired, req["timestamp_end"].to_numpy() + LAG, side="right")
    return int((hi > lo).sum())

T0 = {run: requests[run]["timestamp_start"].min() for run in RUNS}   # to align different runs

rows = []
for run in RUNS:
    real = pair(client[run], service[run])
    controls = [pair(client[run], service[other], shift=T0[run] - T0[other]) for other in RUNS if other != run]
    rows.append({
        "run": run, "client_bursts": len(client[run]), "pairs": len(real),
        "pairs_control": np.mean([len(c) for c in controls]),
        "requests": len(requests[run]), "covered": covered(real, requests[run]),
        "covered_control": np.mean([covered(c, requests[run]) for c in controls]),
    })
res = pd.DataFrame(rows)

tot = res[["client_bursts", "pairs", "pairs_control", "requests", "covered", "covered_control"]].sum()
table = pd.DataFrame({
    "run": list(res["run"]) + ["ALL"],
    "client_bursts": list(res["client_bursts"]) + [tot.client_bursts],
    "paired_%": list((100 * res["pairs"] / res["client_bursts"]).round(1)) + [round(100 * tot.pairs / tot.client_bursts, 1)],
    "control_%": list((100 * res["pairs_control"] / res["client_bursts"]).round(1)) + [round(100 * tot.pairs_control / tot.client_bursts, 1)],
    "requests_covered_%": list((100 * res["covered"] / res["requests"]).round(1)) + [round(100 * tot.covered / tot.requests, 1)],
    "requests_covered_control_%": list((100 * res["covered_control"] / res["requests"]).round(1)) + [round(100 * tot.covered_control / tot.requests, 1)],
})
table.to_csv(RESULTS / "onion_minimal_summary.csv", index=False)
table

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
x, per = np.arange(len(RUNS)), table.iloc[:-1]
for a, real, ctrl, title in [(ax[0], "paired_%", "control_%", "Client bursts paired (%)"),
                             (ax[1], "requests_covered_%", "requests_covered_control_%", "Requests covered (%)")]:
    a.bar(x - 0.2, per[real], 0.4, label="real", color="darkorange")
    a.bar(x + 0.2, per[ctrl], 0.4, label="control (other run)", color="gray")
    a.set_xticks(x, RUNS); a.set_title(title); a.legend()
plt.tight_layout(); plt.show()

## Reading the result

* **real >> control** -> the traffic at the two Guards is linked in time well above chance: an observer on both Guards can associate client and service traffic.
* **real ~ control** -> no usable link in this setup.

Limits: one host and one clock (no network delay), one tiny page, one client and one service (nothing to choose among), thresholds chosen by hand. The percentages are timing statistics, not precision or recall.